In [ ]:
from datasets import load_dataset
# Your script already uses this approach
dataset = load_dataset("fancyzhx/amazon_polarity", split="train[:20000]")

README.md:   0%|          | 0.00/6.81k [00:00<?, ?B/s]

amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  260MB            

amazon_polarity/train-00000-of-00004.par(…): downloading bytes:           |  0.00B            

amazon_polarity/train-00001-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  258MB            

amazon_polarity/train-00001-of-00004.par(…): downloading bytes:           |  0.00B            

amazon_polarity/train-00002-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  255MB            

... (3 outputs left out)

amazon_polarity/test-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B /  117MB            

amazon_polarity/test-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())          # must be True
print("Device:", torch.cuda.get_device_name(0))    # should say Tesla T4
!nvidia-smi --query-gpu=memory.free --format=csv  # free VRAM

CUDA: True
Device: Tesla T4
memory.free [MiB]
14910 MiB


In [ ]:
"""
Case Study 4 — Sentiment Analysis on Amazon Polarity
=====================================================
BERT-base (fp16, GPU) vs BiLSTM baseline.

Fixes included for transformers >= 4.46:
  • TrainingArguments: eval_strategy (not evaluation_strategy)
  • TrainingArguments: warmup_steps (not warmup_ratio)
  • Trainer:           processing_class (not tokenizer)

Expected on T4:  BERT ~13–18 min, LSTM ~4–6 min
Expected accuracy: BERT ~0.92–0.95, LSTM ~0.86–0.90
"""

import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

torch.manual_seed(42)
np.random.seed(42)

# ==================================================================
# SANITY: GPU must be present
# ==================================================================
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, "
    "then Runtime → Disconnect and delete runtime, then re-run this cell."
)
print(f"torch {torch.__version__} | transformers {transformers.__version__} "
      f"| GPU: {torch.cuda.get_device_name(0)}")

# Version-aware flags for transformers >= 4.46 renames
_TR_VER = tuple(map(int, transformers.__version__.split(".")[:2]))
_NEW_API = _TR_VER >= (4, 46)

# ==================================================================
# CONFIG
# ==================================================================
MODEL_NAME   = "bert-base-uncased"
TRAIN_SIZE   = 20_000
TEST_SIZE    = 2_000
MAX_LEN      = 256
MAX_WORDS    = 20_000
EMB_DIM      = 100
BERT_EPOCHS  = 2
BERT_BS      = 32          # reduce to 16 if OOM
BERT_EVAL_BS = 64
FP16         = True        # set False if loss goes NaN

# ==================================================================
# 1. LOAD DATA
# ==================================================================
try:
    train_ds, test_ds
    print("Reusing loaded dataset.")
except NameError:
    print("Loading Amazon Polarity from Hub ...")
    train_ds = load_dataset("fancyzhx/amazon_polarity",
                            split=f"train[:{TRAIN_SIZE}]")
    test_ds  = load_dataset("fancyzhx/amazon_polarity",
                            split=f"test[:{TEST_SIZE}]")

# Trim in case an earlier cell loaded more
train_ds = train_ds.select(range(min(TRAIN_SIZE, len(train_ds))))
test_ds  = test_ds.select(range(min(TEST_SIZE,  len(test_ds))))
print(f"Train: {len(train_ds)}  Test: {len(test_ds)}")

# ==================================================================
# 2. BERT FINE-TUNING
# ==================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok_fn(ex):
    return tokenizer(ex["content"], truncation=True, max_length=MAX_LEN)

train_enc = train_ds.map(tok_fn, batched=True)
test_enc  = test_ds.map(tok_fn,  batched=True)
train_enc.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_enc.set_format("torch",  columns=["input_ids", "attention_mask", "label"])

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).cuda()

def metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy":  accuracy_score(labels, preds),
            "precision": p, "recall": r, "f1": f}

# eval_strategy vs evaluation_strategy
eval_kw = ({"eval_strategy": "epoch"}
           if _TR_VER >= (4, 41)
           else {"evaluation_strategy": "epoch"})

args = TrainingArguments(
    output_dir="./bert_ckpt_gpu",
    learning_rate=2e-5,
    per_device_train_batch_size=BERT_BS,
    per_device_eval_batch_size=BERT_EVAL_BS,
    num_train_epochs=BERT_EPOCHS,
    weight_decay=0.01,
    fp16=FP16,
    save_strategy="no",
    load_best_model_at_end=False,
    report_to="none",
    logging_steps=100,
    warmup_steps=100,          # replaces warmup_ratio (unsupported in some versions)
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    **eval_kw,
)

# Trainer: tokenizer vs processing_class
trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=metrics_fn,
)
if _NEW_API:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("\n>>> Fine-tuning BERT on GPU ...")
t0 = time.time()
trainer.train()
bert_min = (time.time() - t0) / 60

bert_eval = trainer.evaluate()
print(f"\n=== BERT ({bert_min:.1f} min) ===")
for k, v in bert_eval.items():
    if k.startswith("eval_"):
        print(f"  {k:22s}: {v:.4f}")

# ==================================================================
# 3. BiLSTM BASELINE
# ==================================================================
k_tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
k_tok.fit_on_texts(train_ds["content"])

X_train = pad_sequences(k_tok.texts_to_sequences(train_ds["content"]),
                        maxlen=MAX_LEN, padding="post", truncating="post")
X_test  = pad_sequences(k_tok.texts_to_sequences(test_ds["content"]),
                        maxlen=MAX_LEN, padding="post", truncating="post")
y_train = np.array(train_ds["label"])
y_test  = np.array(test_ds["label"])

lstm = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMB_DIM,
              input_length=MAX_LEN, trainable=True),
    Bidirectional(LSTM(64, dropout=0.2)),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])
lstm.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

print("\n>>> Training BiLSTM on GPU ...")
t0 = time.time()
lstm.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
    verbose=1,
)
lstm_min = (time.time() - t0) / 60

lstm_loss, lstm_acc = lstm.evaluate(X_test, y_test, verbose=0)
y_hat = (lstm.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
p, r, f, _ = precision_recall_fscore_support(y_test, y_hat, average="binary")

# ==================================================================
# 4. COMPARISON TABLE
# ==================================================================
tbl = pd.DataFrame({
    "Model":     ["BERT-base (fp16)", "BiLSTM (GPU)"],
    "Accuracy":  [bert_eval["eval_accuracy"],  lstm_acc],
    "Precision": [bert_eval["eval_precision"], p],
    "Recall":    [bert_eval["eval_recall"],    r],
    "F1":        [bert_eval["eval_f1"],        f],
    "Train_min": [bert_min, lstm_min],
})
print("\n=== Model Comparison ===")
print(tbl.round(4).to_string(index=False))
tbl.to_csv("model_comparison_gpu.csv", index=False)
print("Saved model_comparison_gpu.csv")

# ==================================================================
# 5. INFERENCE SPEED (deployment context)
# ==================================================================
sample = test_ds[:500]["content"]

# BERT
model.eval()
t0 = time.time()
with torch.no_grad():
    for txt in sample:
        inp = tokenizer(txt, return_tensors="pt",
                        truncation=True, max_length=MAX_LEN).to("cuda")
        model(**inp)
torch.cuda.synchronize()                    # critical for accurate GPU timing
bert_ms = (time.time() - t0) / len(sample) * 1000

# LSTM
seqs = pad_sequences(k_tok.texts_to_sequences(sample),
                     maxlen=MAX_LEN, padding="post", truncating="post")
t0 = time.time()
lstm.predict(seqs, batch_size=1, verbose=0)
lstm_ms = (time.time() - t0) / len(sample) * 1000

print(f"\nInference/sample (GPU): BERT={bert_ms:.1f} ms   BiLSTM={lstm_ms:.1f} ms")
print(f"Speed ratio: {bert_ms / lstm_ms:.2f}×")

# ==================================================================
# 6. DISCUSSION
# ==================================================================
print("\n--- Transfer-learning efficiency ---")
print(f"BERT converged in {BERT_EPOCHS} epochs ({bert_min:.1f} min) → "
      f"{bert_eval['eval_accuracy']:.3f} acc. "
      f"BiLSTM needed 5 epochs ({lstm_min:.1f} min) → {lstm_acc:.3f} acc.")
print("\n--- Deployment constraints ---")
print("BERT-base ≈110M params → high GPU memory (~440 MB weights, "
      "~4–8 GB activations at bs=32). Quantisation (INT8) cuts size ~4×, "
      "distillation (DistilBERT) cuts ~40%. Both are relevant trade-offs "
      "for production deployment.")

torch 2.11.0+cu130 | transformers 5.21.0 | GPU: Tesla T4
Reusing loaded dataset.
Train: 20000  Test: 2000


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



>>> Fine-tuning BERT on GPU ...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.205759,0.199191,0.924000,0.942574,0.910134,0.926070
2,0.119779,0.214827,0.930500,0.932316,0.934990,0.933652


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.119779,0.214827,2,0.930500,0.932316,0.934990,0.933652



=== BERT (6.3 min) ===
  eval_loss             : 0.2148
  eval_accuracy         : 0.9305
  eval_precision        : 0.9323
  eval_recall           : 0.9350
  eval_f1               : 0.9337

>>> Training BiLSTM on GPU ...
Epoch 1/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 14s 32ms/step - accuracy: 0.7248 - loss: 0.5229 - val_accuracy: 0.7965 - val_loss: 0.4452
Epoch 2/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.8851 - loss: 0.2909 - val_accuracy: 0.8315 - val_loss: 0.3991
Epoch 3/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9328 - loss: 0.1893 - val_accuracy: 0.8300 - val_loss: 0.4377
Epoch 4/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.9488 - loss: 0.1510 - val_accuracy: 0.8190 - val_loss: 0.5722

=== Model Comparison ===
           Model  Accuracy  Precision  Recall     F1  Train_min
BERT-base (fp16)    0.9305     0.9323  0.9350 0.9337     6.3380
    BiLSTM (GPU)    0.8530     0.8658  0.8509 0.8582     0.4293
Saved model_comparison_gpu.csv

Inference/sample